# Clase 05 — Pipelines reproducibles

**Curso:** MLOps y Monitorización con Python · **Bloque 2:** Versionar y medir
**Tiempo estimado:** 105 minutos · **Capítulo del manual:** `manual/05_pipelines_reproducibles.md`

## Qué vas a ver en esta clase

- Construir una `Pipeline` de scikit-learn **completa** (limpieza, `ColumnTransformer` para columnas numéricas/categóricas, modelo) que se entrena y se aplica con **una sola llamada**.
- Un caso real de **fuga de información** (sobremuestrear antes de dividir los datos): comparamos la métrica optimista con la honesta, con números concretos.
- **Orquestación ligera**: cuatro funciones de Python (`preparar`, `entrenar`, `evaluar`, `registrar`) llamadas desde un único script de entrada, `main.py`, ejecutado con `subprocess.run`.
- Qué es un **feature store**, a nivel de concepto, y por qué este curso no lo implementa.
- El **sesgo entrenamiento-servicio** (*training-serving skew*), con un ejemplo pequeño y verificable, y su relación con el feature store.

Todo ocurre dentro de una carpeta de trabajo efímera (`proyecto_churn/`), creada al principio y **eliminada al final** de este mismo notebook.

## Requisitos

- `pandas` y `scikit-learn` (la celda siguiente instala solo lo que falte, también en Google Colab).
- La carpeta `datos/` del curso, con `churn.csv` dentro, y haber leído `datos/LEEME.md`.
- De la clase 4: la configuración 3 (regresión logística, `class_weight="balanced"`, `C=1,0`), que reutilizamos aquí.

> **Cómo usar este notebook:** ejecuta las celdas en orden, de arriba a abajo, una sola vez. Tarda menos de un minuto en total.

---
## Instalación

Instala solo lo que falte.

In [ ]:
import subprocess
import sys


def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)


asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("scikit-learn", "sklearn")
asegurar("joblib", "joblib")
print("Librerías listas.")

---
## Celda 0 — Setup

Importamos las librerías, definimos `cargar_datos` y fijamos `SEMILLA = 42` para toda la clase.

In [ ]:
import os
import shutil
import subprocess
import sys
import json

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/18-mlops-monitoraggio/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

SEMILLA = 42
COLUMNAS_CATEGORICAS = ["plan", "contrato", "metodo_pago"]
COLUMNAS_NUMERICAS = ["antiguedad_meses", "uso_mensual_horas", "tickets_soporte", "factura_mensual"]

print("Librerías y funciones auxiliares listas.")

---
## Carpeta de trabajo efímera

Como en las clases anteriores: creamos `proyecto_churn/` desde cero.

In [ ]:
CARPETA = "proyecto_churn"

# Limpiamos y recreamos la carpeta para que el notebook sea repetible desde cero
if os.path.exists(CARPETA):
    shutil.rmtree(CARPETA)
os.makedirs(CARPETA)
print("Carpeta de trabajo creada:", os.path.abspath(CARPETA))

---
## 1. Cargar los datos y dividir train/test

Cargamos `churn.csv` y dividimos ANTES de construir ningún paso que aprenda de los datos.

In [ ]:
RUTA_CSV = cargar_datos("churn.csv")
df = pd.read_csv(RUTA_CSV)

X = df[COLUMNAS_CATEGORICAS + COLUMNAS_NUMERICAS]
y = df["abandono"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEMILLA, stratify=y
)

print("filas totales:", len(df))
print("filas de entrenamiento / prueba:", len(X_train), "/", len(X_test))
assert len(X_train) + len(X_test) == len(df)

---
## 2. La Pipeline completa: limpieza + ColumnTransformer + modelo

Una única `Pipeline` encadena: imputación (limpieza), `ColumnTransformer` (una rama para categóricas con `OneHotEncoder`, otra para numéricas con `StandardScaler`) y el modelo (regresión logística balanceada). `pipeline.fit(X_train, y_train)` entrena TODOS los pasos con una sola llamada.

In [ ]:
def construir_pipeline():
    """Limpieza (imputar) + ColumnTransformer (categóricas/numéricas) + modelo, en un único objeto."""
    preprocesado = ColumnTransformer([
        ("categoricas", Pipeline([
            ("imputar", SimpleImputer(strategy="most_frequent")),
            ("codificar", OneHotEncoder(handle_unknown="ignore")),
        ]), COLUMNAS_CATEGORICAS),
        ("numericas", Pipeline([
            ("imputar", SimpleImputer(strategy="median")),
            ("escalar", StandardScaler()),
        ]), COLUMNAS_NUMERICAS),
    ])
    return Pipeline([
        ("preprocesado", preprocesado),
        ("modelo", LogisticRegression(class_weight="balanced", random_state=SEMILLA, max_iter=1000)),
    ])


# Una única llamada entrena TODOS los pasos (imputadores, escalador, codificador, modelo)
pipeline = construir_pipeline()
pipeline.fit(X_train, y_train)

nombres_columnas = list(pipeline.named_steps["preprocesado"].get_feature_names_out())
print("columnas que ve el modelo tras el preprocesado:", len(nombres_columnas))
print(nombres_columnas)
assert len(nombres_columnas) == 12

**¿Qué observas?** El modelo ve 12 columnas: 8 del `OneHotEncoder` (3 `plan` + 2 `contrato` + 3 `metodo_pago`) y 4 numéricas escaladas — se puede contar a mano en `COLUMNAS_CATEGORICAS`/`COLUMNAS_NUMERICAS` y comprobarlo con el `assert`.

In [ ]:
# Una única llamada aplica TODOS los pasos ya ajustados, en el mismo orden
predicciones = pipeline.predict(X_test)
probabilidades = pipeline.predict_proba(X_test)[:, 1]

exactitud = round(accuracy_score(y_test, predicciones), 4)
f1 = round(f1_score(y_test, predicciones), 4)
auc = round(roc_auc_score(y_test, probabilidades), 4)

print("exactitud:", exactitud)
print("f1:", f1)
print("auc:", auc)
assert (exactitud, f1, auc) == (0.6547, 0.3166, 0.7227), "deberían coincidir con la configuración 3 de la clase 4"

**¿Qué observas?** `pipeline.predict(X_test)` aplica, con una sola llamada, TODOS los pasos ya ajustados (imputadores, escalador, codificador, modelo). Las cifras (exactitud 0,6547, F1 0,3166, AUC 0,7227) coinciden con la configuración 3 de la clase 4, porque es la misma receta.

---
### 2.2 Por qué la limpieza va siempre, aunque hoy no haga nada

`churn.csv` no tiene huecos, pero corrompemos a propósito una fila (un valor ausente real) y comparamos la Pipeline con `SimpleImputer` frente a una versión sin él.

In [ ]:
# Corrompemos a propósito una fila del conjunto de prueba (un valor ausente real)
X_con_hueco = X_test.copy()
X_con_hueco.iloc[0, X_con_hueco.columns.get_loc("uso_mensual_horas")] = np.nan

prediccion_con_imputer = pipeline.predict(X_con_hueco.iloc[[0]])
print("con SimpleImputer, predicción de la fila con NaN:", prediccion_con_imputer)

# Pipeline SIN imputador, para comparar (misma receta, sin el paso "imputar")
preprocesado_sin_imputar = ColumnTransformer([
    ("categoricas", OneHotEncoder(handle_unknown="ignore"), COLUMNAS_CATEGORICAS),
    ("numericas", StandardScaler(), COLUMNAS_NUMERICAS),
])
pipeline_sin_imputar = Pipeline([
    ("preprocesado", preprocesado_sin_imputar),
    ("modelo", LogisticRegression(class_weight="balanced", random_state=SEMILLA, max_iter=1000)),
])
pipeline_sin_imputar.fit(X_train, y_train)

try:
    pipeline_sin_imputar.predict(X_con_hueco.iloc[[0]])
    fallo = False
except ValueError as error:
    fallo = True
    print("sin SimpleImputer, ValueError:", str(error)[:40] + "...")

assert fallo, "sin imputador, la Pipeline debería fallar con un valor ausente"

**¿Qué observas?** Con `SimpleImputer`, la fila con `NaN` se predice con normalidad. Sin él, la misma fila lanza `ValueError: Input X contains NaN` y detiene todo el flujo. La limpieza no es un adorno: decide si un solo valor ausente en producción rompe una predicción o pasa desapercibido.

---
## 3. Un caso real de fuga de información: sobremuestrear antes de dividir

Ya viste la fuga de información, con otro mecanismo, en el curso de NLP clásico (clase 5). Aquí usamos un mecanismo distinto y también real: duplicar la clase minoritaria (`abandono=1`) ANTES de dividir en entrenamiento/prueba, para "balancear" el dataset. El problema: copias casi idénticas de una fila de entrenamiento pueden terminar en el conjunto de prueba.

In [ ]:
FACTOR_SOBREMUESTREO = 7


def sobremuestrear(df_en, semilla):
    """Duplica la clase minoritaria (abandono=1) con un poco de ruido, FACTOR_SOBREMUESTREO veces."""
    rng = np.random.RandomState(semilla)
    minoria = df_en[df_en["abandono"] == 1]
    copias = []
    for _ in range(FACTOR_SOBREMUESTREO):
        copia = minoria.copy()
        for columna in COLUMNAS_NUMERICAS:
            copia[columna] = copia[columna] * (1 + rng.normal(0, 0.01, size=len(copia)))
        copias.append(copia)
    return pd.concat([df_en] + copias, ignore_index=True)

In [ ]:
# ENFOQUE OPTIMISTA (con fuga): sobremuestreamos TODO el dataset ANTES de dividir
df_fugado = sobremuestrear(df, SEMILLA)
print("filas tras sobremuestrear todo el dataset:", len(df_fugado))

X_fugado = df_fugado[COLUMNAS_CATEGORICAS + COLUMNAS_NUMERICAS]
y_fugado = df_fugado["abandono"]
X_fug_train, X_fug_test, y_fug_train, y_fug_test = train_test_split(
    X_fugado, y_fugado, test_size=0.25, random_state=SEMILLA, stratify=y_fugado
)

pipeline_optimista = construir_pipeline()
pipeline_optimista.fit(X_fug_train, y_fug_train)
pred_optimista = pipeline_optimista.predict(X_fug_test)
proba_optimista = pipeline_optimista.predict_proba(X_fug_test)[:, 1]

exactitud_optimista = round(accuracy_score(y_fug_test, pred_optimista), 4)
f1_optimista = round(f1_score(y_fug_test, pred_optimista), 4)
auc_optimista = round(roc_auc_score(y_fug_test, proba_optimista), 4)

print("(optimista) filas train/test:", len(X_fug_train), "/", len(X_fug_test))
print("(optimista) exactitud/f1/auc:", exactitud_optimista, f1_optimista, auc_optimista)

**¿Qué observas?** Tras sobremuestrear TODO el dataset, se divide en train/test — pero varias copias casi idénticas de la misma fila original pueden caer una en train y otra en test. El modelo "reconoce" esas filas en vez de predecirlas de verdad.

In [ ]:
# ENFOQUE HONESTO: dividimos primero con los datos originales, sobremuestreamos SOLO el entrenamiento
train_honesto = X_train.copy()
train_honesto["abandono"] = y_train.values
train_honesto_aumentado = sobremuestrear(train_honesto, SEMILLA)

X_hon_train = train_honesto_aumentado[COLUMNAS_CATEGORICAS + COLUMNAS_NUMERICAS]
y_hon_train = train_honesto_aumentado["abandono"]

pipeline_honesto = construir_pipeline()
pipeline_honesto.fit(X_hon_train, y_hon_train)
# El conjunto de prueba (X_test, y_test) es el ORIGINAL, intacto: nunca pasó por sobremuestrear()
pred_honesta = pipeline_honesto.predict(X_test)
proba_honesta = pipeline_honesto.predict_proba(X_test)[:, 1]

exactitud_honesta = round(accuracy_score(y_test, pred_honesta), 4)
f1_honesto = round(f1_score(y_test, pred_honesta), 4)
auc_honesto = round(roc_auc_score(y_test, proba_honesta), 4)

print("(honesto) filas train aumentado / test intacto:", len(X_hon_train), "/", len(X_test))
print("(honesto) exactitud/f1/auc:", exactitud_honesta, f1_honesto, auc_honesto)

veces_inflado = round(f1_optimista / f1_honesto, 2)
print("el F1 optimista es, veces el honesto:", veces_inflado)
assert veces_inflado > 1.5, "la fuga debería inflar claramente el F1"

**¿Qué observas?** El F1 optimista (con fuga) es 2,14 veces el F1 honesto: parece el doble de bueno de lo que realmente es. El AUC apenas cambia (0,7207 frente a 0,7229): compara el orden global de las probabilidades, algo que las copias casi idénticas no alteran mucho; el F1, con un umbral fijo sobre predicciones concretas, sí nota de lleno el reconocimiento. Ninguna métrica sola basta para confiar en un resultado.

---
## 4. Orquestación ligera: `main.py` con `preparar → entrenar → evaluar → registrar`

Escribimos un script real, `proyecto_churn/main.py`, con cuatro funciones (una por fase) y una función `main()` que las llama en orden. Lo ejecutamos con `subprocess.run`, exactamente como se ejecutaría desde una terminal.

In [ ]:
MAIN_PY = """
import os
import sys
import json
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

COLUMNAS_CATEGORICAS = ["plan", "contrato", "metodo_pago"]
COLUMNAS_NUMERICAS = ["antiguedad_meses", "uso_mensual_horas", "tickets_soporte", "factura_mensual"]
SEMILLA = 42
CARPETA = "proyecto_churn"


def cargar_datos(nombre):
    '''Busca un archivo de datos en datos/, ../datos/ y /content/datos/.'''
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/18-mlops-monitoraggio/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError("No encuentro " + nombre)


def preparar():
    '''Fase 1: carga churn.csv y separa entrenamiento/prueba.'''
    print("[preparar] cargando churn.csv y dividiendo train/test...")
    ruta = cargar_datos("churn.csv")
    df = pd.read_csv(ruta)
    X = df[COLUMNAS_CATEGORICAS + COLUMNAS_NUMERICAS]
    y = df["abandono"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEMILLA, stratify=y)
    print(f"[preparar] filas train={len(X_train)} test={len(X_test)}")
    return X_train, X_test, y_train, y_test


def construir_pipeline():
    '''Misma receta que en el notebook: limpieza + ColumnTransformer + modelo.'''
    preprocesado = ColumnTransformer([
        ("categoricas", Pipeline([
            ("imputar", SimpleImputer(strategy="most_frequent")),
            ("codificar", OneHotEncoder(handle_unknown="ignore")),
        ]), COLUMNAS_CATEGORICAS),
        ("numericas", Pipeline([
            ("imputar", SimpleImputer(strategy="median")),
            ("escalar", StandardScaler()),
        ]), COLUMNAS_NUMERICAS),
    ])
    return Pipeline([
        ("preprocesado", preprocesado),
        ("modelo", LogisticRegression(class_weight="balanced", random_state=SEMILLA, max_iter=1000)),
    ])


def entrenar(X_train, y_train):
    '''Fase 2: ajusta la Pipeline completa con una sola llamada.'''
    print("[entrenar] ajustando la Pipeline completa (preprocesado + modelo)...")
    pipeline = construir_pipeline()
    pipeline.fit(X_train, y_train)
    print("[entrenar] pipeline.fit() completado.")
    return pipeline


def evaluar(pipeline, X_test, y_test):
    '''Fase 3: aplica la Pipeline al conjunto de prueba con una sola llamada.'''
    print("[evaluar] calculando métricas en el conjunto de prueba...")
    predicciones = pipeline.predict(X_test)
    probabilidades = pipeline.predict_proba(X_test)[:, 1]
    metricas = {
        "exactitud": round(accuracy_score(y_test, predicciones), 4),
        "f1": round(f1_score(y_test, predicciones), 4),
        "auc": round(roc_auc_score(y_test, probabilidades), 4),
    }
    print("[evaluar] metricas:", metricas)
    return metricas


def registrar(pipeline, metricas):
    '''Fase 4: guarda el modelo entrenado y sus métricas en la carpeta efímera.'''
    print("[registrar] guardando el modelo y sus metadatos...")
    ruta_modelo = os.path.join(CARPETA, "modelo.joblib")
    joblib.dump(pipeline, ruta_modelo)
    ruta_metricas = os.path.join(CARPETA, "metricas.json")
    with open(ruta_metricas, "w") as f:
        json.dump(metricas, f, indent=2)
    print(f"[registrar] modelo guardado en {ruta_modelo}")
    print(f"[registrar] metricas guardadas en {ruta_metricas}")


def main():
    X_train, X_test, y_train, y_test = preparar()
    pipeline = entrenar(X_train, y_train)
    metricas = evaluar(pipeline, X_test, y_test)
    registrar(pipeline, metricas)
    print("[main] orquestacion completa: preparar -> entrenar -> evaluar -> registrar.")


if __name__ == "__main__":
    main()
"""

In [ ]:
ruta_main = os.path.join(CARPETA, "main.py")
with open(ruta_main, "w", encoding="utf-8") as f:
    f.write(MAIN_PY)
print("script de entrada creado en:", ruta_main)

In [ ]:
# Ejecutamos el script como se haría desde una terminal o desde un flujo de CI/CD
resultado = subprocess.run([sys.executable, ruta_main], capture_output=True, text=True)
print(resultado.stdout)
if resultado.returncode != 0:
    print(resultado.stderr)
assert resultado.returncode == 0, "el script main.py debería terminar sin errores"

**¿Qué observas?** La salida muestra las cuatro fases en orden (`[preparar]`, `[entrenar]`, `[evaluar]`, `[registrar]`), tal como las escribimos en `main.py`. El script termina con código de salida 0 (sin errores), igual que comprobaría una CI/CD real.

In [ ]:
with open(os.path.join(CARPETA, "metricas.json")) as f:
    metricas_script = json.load(f)
print("métricas leídas del archivo registrado por main.py:", metricas_script)

assert metricas_script == {"exactitud": exactitud, "f1": f1, "auc": auc}, "deberían coincidir con las calculadas a mano en la sección 2"
assert os.path.exists(os.path.join(CARPETA, "modelo.joblib"))
print("coinciden con las de la sección 2: es la misma Pipeline, ejecutada dos veces con la misma semilla.")

---
## 5. Feature store: concepto, y por qué aquí no lo implementamos

Un **feature store** calcula, guarda y sirve las variables de un modelo una sola vez, para que entrenamiento y producción lean siempre el mismo valor. No lo implementamos en este curso: requiere infraestructura propia (base de datos compartida, servicio en tiempo real) que no cabe en un entorno local/Colab con un solo proyecto. El capítulo del manual (sección 5) tiene la tabla completa de comparación.

---
## 6. Sesgo entrenamiento-servicio: un ejemplo pequeño y verificable

Simulamos dos reimplementaciones de la misma variable (`antiguedad_meses` agrupada en categorías): una en el código de "entrenamiento" y otra, con un error de límite (`<` en vez de `<=`), en el código de "servicio".

In [ ]:
def categoria_entrenamiento(meses):
    """Código de entrenamiento: 'medio' incluye el límite superior (<=36)."""
    if meses < 12:
        return "nuevo"
    elif meses <= 36:
        return "medio"
    return "antiguo"


def categoria_servicio(meses):
    """Reimplementación en el servicio, con un error de límite (<36 en vez de <=36)."""
    if meses < 12:
        return "nuevo"
    elif meses < 36:
        return "medio"
    return "antiguo"


categoria_train = df["antiguedad_meses"].apply(categoria_entrenamiento)
categoria_serv = df["antiguedad_meses"].apply(categoria_servicio)
filas_distintas = int((categoria_train != categoria_serv).sum())
porcentaje = round(filas_distintas / len(df) * 100, 2)

print("filas con categoría distinta entre entrenamiento y servicio:", filas_distintas, "/", len(df), f"({porcentaje}%)")
assert filas_distintas == 50

**¿Qué observas?** Solo 50 filas de 3 000 (1,67 %) tienen `antiguedad_meses == 36` exactamente, y son las únicas donde las dos funciones difieren. Ningún error, ninguna excepción: un sesgo silencioso que solo se detecta comparando ambas implementaciones fila por fila. Un feature store elimina este riesgo porque la variable se calcula en un único lugar.

---
## 7. Limpieza final

Eliminamos `proyecto_churn/` (con `main.py`, el modelo y las métricas) para que no quede ningún residuo dentro del repositorio del curso.

In [ ]:
shutil.rmtree(CARPETA)
print("Carpeta de trabajo eliminada:", CARPETA)
assert not os.path.exists(CARPETA)

---
## Resumen de la clase

- Una `Pipeline` de scikit-learn completa (limpieza + `ColumnTransformer` + modelo) se entrena y se aplica con una sola llamada cada vez, sin repetir pasos a mano.
- La fuga de información también aparece al sobremuestrear antes de dividir: el F1 "optimista" fue **2,14 veces** el honesto, aunque el AUC casi no cambió — el mismo tipo de problema visto en NLP clase 5, con otro mecanismo.
- La **orquestación ligera** (`preparar → entrenar → evaluar → registrar`, en `main.py`) cubre lo esencial de un flujo de trabajo sin necesitar un orquestador pesado.
- Un **feature store** evita que la misma variable se reimplemente en varios sitios; sin él, el **sesgo entrenamiento-servicio** puede aparecer con un simple error de límite, como muestra el ejemplo del 1,67 % de filas afectadas.

*Continúa con la clase 6: Probar los sistemas de ML.*